In [ ]:
# ==========================================================
# Cell 1 : Import Libraries
# ==========================================================

import os
import json
import random
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print("="*60)
print("PyTorch Version :", torch.__version__)
print("="*60)

In [ ]:
# ==========================================================
# Cell 2 : GPU Configuration
# ==========================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("="*60)
print("Device :", device)

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

torch.backends.cudnn.benchmark = True

print("="*60)

In [ ]:
# ==========================================================
# Cell 3 : Reproducibility
# ==========================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed :", SEED)

In [ ]:
# ==========================================================
# Cell 4 : Dataset Paths
# ==========================================================

ROOT = "/kaggle/input/datasets/solesensei/solesensei_bdd100k"

IMAGE_ROOT = os.path.join(
    ROOT,
    "bdd100k",
    "bdd100k",
    "images"
)

LABEL_FILE = os.path.join(
    ROOT,
    "bdd100k_labels_release",
    "bdd100k",
    "labels",
    "bdd100k_labels_images_train.json"
)

print("Image Root :", IMAGE_ROOT)
print("Label File :", LABEL_FILE)

In [ ]:
# ==========================================================
# Cell 5 : Verify Dataset
# ==========================================================

print("="*60)

print(
    "Images Folder Exists :",
    os.path.exists(IMAGE_ROOT)
)

print(
    "JSON Exists :",
    os.path.exists(LABEL_FILE)
)

print("="*60)

In [ ]:
# ==========================================================
# Cell 6 : Read Metadata
# ==========================================================

with open(LABEL_FILE,"r") as f:

    metadata = json.load(f)

metadata = pd.DataFrame(metadata)

print(metadata.shape)

metadata.head()

In [ ]:
# ==========================================================
# Cell 7 : Extract Weather Information
# ==========================================================

metadata["weather"] = metadata["attributes"].apply(
    lambda x: x["weather"]
)

weather_df = metadata[
    ["name","weather"]
].copy()

weather_df.columns = [
    "image",
    "weather"
]

weather_df.head()

In [ ]:
# ==========================================================
# Cell 8 : Weather Distribution
# ==========================================================

print(
    weather_df["weather"].value_counts()
)

plt.figure(figsize=(8,5))

weather_df["weather"].value_counts().plot(
    kind="bar"
)

plt.title("Weather Distribution")

plt.show()

In [ ]:
# ==========================================================
# Cell 9 : Remove Undefined Samples
# ==========================================================

weather_df = weather_df[
    weather_df.weather != "undefined"
]

weather_df = weather_df.reset_index(
    drop=True
)

print(weather_df.shape)

print()

print(
    weather_df.weather.value_counts()
)

In [ ]:
# ==========================================================
# Cell 10 : Label Encoding
# ==========================================================

encoder = LabelEncoder()

weather_df["label"] = encoder.fit_transform(
    weather_df["weather"]
)

print("="*60)

print("Weather Classes")

for i,c in enumerate(encoder.classes_):

    print(i,"->",c)

print("="*60)

weather_classes = list(
    encoder.classes_
)

In [ ]:
# ==========================================================
# Cell 11 : Build Image Index
# ==========================================================

image_index = {}

for img in tqdm(
    Path(IMAGE_ROOT).rglob("*.jpg")
):

    image_index[img.name] = str(img)

print()

print("="*60)

print(
    "Indexed Images :",
    len(image_index)
)

print("="*60)

In [ ]:
# ==========================================================
# Cell 12 : Map Image Paths
# ==========================================================

weather_df["image_path"] = weather_df["image"].map(
    image_index
)

print("="*60)

print(
    "Missing Images :",
    weather_df["image_path"].isna().sum()
)

print("="*60)

weather_df = weather_df.dropna(
    subset=["image_path"]
).reset_index(drop=True)

weather_df.head()

In [ ]:
# ==========================================================
# Cell 13 : Train Validation Split
# ==========================================================

train_df, val_df = train_test_split(

    weather_df,

    test_size=0.20,

    random_state=SEED,

    stratify=weather_df["label"]

)

train_df = train_df.reset_index(drop=True)

val_df = val_df.reset_index(drop=True)

print()

print(
    "Training Images :",
    len(train_df)
)

print(
    "Validation Images :",
    len(val_df)
)

In [ ]:
# ==========================================================
# Cell 14 : Weather Classes
# ==========================================================

print(weather_classes)

In [ ]:
# ==========================================================
# Cell 15 : Configuration
# ==========================================================

IMAGE_SIZE = 224

BATCH_SIZE = 64

NUM_WORKERS = 2

PIN_MEMORY = torch.cuda.is_available()

print("="*60)
print("Image Size :", IMAGE_SIZE)
print("Batch Size :", BATCH_SIZE)
print("Workers :", NUM_WORKERS)
print("="*60)

In [ ]:
# ==========================================================
# Cell 16 : Image Transformations
# ==========================================================

train_transform = transforms.Compose([

    transforms.ToPILImage(),

    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.RandomHorizontalFlip(0.5),

    transforms.RandomRotation(10),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),

    transforms.RandomAffine(
        degrees=0,
        translate=(0.05,0.05)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )

])

val_transform = transforms.Compose([

    transforms.ToPILImage(),

    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )

])

print("Transforms Ready")

In [ ]:
# ==========================================================
# Cell 17 : Weather Dataset
# ==========================================================

class WeatherDataset(Dataset):

    def __init__(self, dataframe, transform=None):

        self.df = dataframe.reset_index(drop=True)

        self.transform = transform

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        img_path = self.df.loc[idx, "image_path"]

        label = int(self.df.loc[idx, "label"])

        image = cv2.imread(img_path)

        if image is None:

            raise FileNotFoundError(
                f"Unable to read image:\n{img_path}"
            )

        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        if self.transform:

            image = self.transform(image)

        return image, label

In [ ]:
# ==========================================================
# Cell 18 : Dataset Objects
# ==========================================================

train_dataset = WeatherDataset(
    train_df,
    train_transform
)

val_dataset = WeatherDataset(
    val_df,
    val_transform
)

print("="*60)

print("Training Samples :", len(train_dataset))

print("Validation Samples :", len(val_dataset))

print("="*60)

In [ ]:
# ==========================================================
# Cell 19 : Test Dataset
# ==========================================================

image, label = train_dataset[0]

print("="*60)

print("Image Shape :", image.shape)

print("Weather :", weather_classes[label])

print("="*60)

In [ ]:
# ==========================================================
# Cell 20 : DataLoader
# ==========================================================

train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=NUM_WORKERS,

    pin_memory=PIN_MEMORY,

    drop_last=True

)

val_loader = DataLoader(

    val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=PIN_MEMORY

)

print("="*60)

print("Train Batches :", len(train_loader))

print("Validation Batches :", len(val_loader))

print("="*60)

In [ ]:
# ==========================================================
# Cell 21 : Batch Test
# ==========================================================

images, labels = next(iter(train_loader))

print("="*60)

print("Images :", images.shape)

print("Labels :", labels.shape)

print("="*60)

In [ ]:
# ==========================================================
# Cell 22 : Visualize Batch
# ==========================================================

plt.figure(figsize=(12,12))

for i in range(9):

    image = images[i].permute(1,2,0).numpy()

    image = image * np.array([0.229,0.224,0.225])

    image = image + np.array([0.485,0.456,0.406])

    image = np.clip(image,0,1)

    plt.subplot(3,3,i+1)

    plt.imshow(image)

    plt.title(weather_classes[labels[i]])

    plt.axis("off")

plt.tight_layout()

plt.show()

In [ ]:
# ==========================================================
# Cell 23 : Dataset Statistics
# ==========================================================

print("="*60)

print("Training Images :", len(train_dataset))

print("Validation Images :", len(val_dataset))

print("Number of Classes :", len(weather_classes))

print()

for cls in weather_classes:

    count = (train_df["weather"] == cls).sum()

    print(f"{cls:20s} : {count}")

print("="*60)

In [ ]:
# ==========================================================
# Cell 24 : Final Check
# ==========================================================

print("="*60)

print("Dataset Ready")

print("Weather Classes :", weather_classes)

print("Input Size :", IMAGE_SIZE)

print("Batch Size :", BATCH_SIZE)

print("="*60)

In [ ]:
# ==========================================================
# Cell 25 : Conv Block
# ==========================================================

class ConvBlock(nn.Module):

    def __init__(self, in_channels, out_channels):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(out_channels),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(out_channels),

            nn.ReLU(inplace=True)

        )

    def forward(self, x):

        return self.block(x)

In [ ]:
# ==========================================================
# Cell 26 : Residual Block
# ==========================================================

class ResidualBlock(nn.Module):

    def __init__(self, channels):

        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv2d(
                channels,
                channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(channels),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                channels,
                channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(channels)

        )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):

        out = self.conv(x)

        out += x

        out = self.relu(out)

        return out

In [ ]:
# ==========================================================
# Cell 27 : SE Attention
# ==========================================================

class SEBlock(nn.Module):

    def __init__(self, channels, reduction=16):

        super().__init__()

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.fc = nn.Sequential(

            nn.Linear(
                channels,
                channels // reduction
            ),

            nn.ReLU(inplace=True),

            nn.Linear(
                channels // reduction,
                channels
            ),

            nn.Sigmoid()

        )

    def forward(self, x):

        b, c, _, _ = x.size()

        y = self.pool(x).view(b, c)

        y = self.fc(y).view(b, c, 1, 1)

        return x * y

In [ ]:
# ==========================================================
# Cell 28 : Feature Block
# ==========================================================

class FeatureBlock(nn.Module):

    def __init__(self, in_channels, out_channels):

        super().__init__()

        self.conv = ConvBlock(
            in_channels,
            out_channels
        )

        self.res = ResidualBlock(
            out_channels
        )

        self.se = SEBlock(
            out_channels
        )

        self.pool = nn.MaxPool2d(2)

    def forward(self, x):

        x = self.conv(x)

        x = self.res(x)

        x = self.se(x)

        x = self.pool(x)

        return x

In [ ]:
# ==========================================================
# Cell 28 : Feature Block
# ==========================================================

class FeatureBlock(nn.Module):

    def __init__(self, in_channels, out_channels):

        super().__init__()

        self.conv = ConvBlock(
            in_channels,
            out_channels
        )

        self.res = ResidualBlock(
            out_channels
        )

        self.se = SEBlock(
            out_channels
        )

        self.pool = nn.MaxPool2d(2)

    def forward(self, x):

        x = self.conv(x)

        x = self.res(x)

        x = self.se(x)

        x = self.pool(x)

        return x

In [ ]:
# ==========================================================
# Cell 29 : WeatherNet
# ==========================================================

class WeatherNet(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        self.stage1 = FeatureBlock(3, 32)

        self.stage2 = FeatureBlock(32, 64)

        self.stage3 = FeatureBlock(64, 128)

        self.stage4 = FeatureBlock(128, 256)

        self.global_pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Dropout(0.5),

            nn.Linear(256,128),

            nn.ReLU(inplace=True),

            nn.Dropout(0.3),

            nn.Linear(128,num_classes)

        )

    def forward(self, x):

        x = self.stage1(x)

        x = self.stage2(x)

        x = self.stage3(x)

        x = self.stage4(x)

        x = self.global_pool(x)

        x = self.classifier(x)

        return x

In [ ]:
# ==========================================================
# Cell 30 : Create Model
# ==========================================================

model = WeatherNet(
    num_classes=len(weather_classes)
).to(device)

print(model)

In [ ]:
# ==========================================================
# Cell 31 : Parameters
# ==========================================================

total = sum(p.numel() for p in model.parameters())

trainable = sum(

    p.numel()

    for p in model.parameters()

    if p.requires_grad

)

print("="*60)

print("Total Parameters :", f"{total:,}")

print("Trainable Parameters :", f"{trainable:,}")

print("="*60)

In [ ]:
# ==========================================================
# Cell 32 : Forward Pass
# ==========================================================

images, labels = next(iter(train_loader))

images = images.to(device)

outputs = model(images)

print("="*60)

print("Input Shape :", images.shape)

print("Output Shape :", outputs.shape)

print("="*60)

In [ ]:
# ==========================================================
# Cell 33 : Summary
# ==========================================================

print("="*60)

print(model)

print("="*60)

In [ ]:
# ==========================================================
# Cell 34 : Loss Function
# ==========================================================

criterion = nn.CrossEntropyLoss()

print("Loss Function Ready")

In [ ]:
# ==========================================================
# Cell 35 : Optimizer
# ==========================================================

optimizer = optim.AdamW(

    model.parameters(),

    lr=1e-3,

    weight_decay=1e-4

)

print("Optimizer Ready")

In [ ]:
# ==========================================================
# Cell 36 : Scheduler
# ==========================================================

scheduler = optim.lr_scheduler.ReduceLROnPlateau(

    optimizer,

    mode="min",

    factor=0.5,

    patience=2

)

print("Scheduler Ready")

In [ ]:
# ==========================================================
# Cell 37 : Mixed Precision
# ==========================================================

scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

print("AMP Enabled :", torch.cuda.is_available())

In [ ]:
# ==========================================================
# Cell 38 : Training Configuration
# ==========================================================

EPOCHS = 20

best_accuracy = 0.0

history = {
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": []
}

print("="*60)
print("Epochs :", EPOCHS)
print("="*60)

In [ ]:
# ==========================================================
# Cell 39 : Train One Epoch
# ==========================================================

def train_one_epoch(model, loader):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    loop = tqdm(loader, leave=False)

    for images, labels in loop:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast(
            device_type="cuda",
            enabled=torch.cuda.is_available()
        ):

            outputs = model(images)

            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        running_loss += loss.item()

        predictions = outputs.argmax(1)

        correct += (predictions == labels).sum().item()

        total += labels.size(0)

        loop.set_postfix(
            loss=loss.item(),
            acc=100 * correct / total
        )

    epoch_loss = running_loss / len(loader)

    epoch_acc = 100 * correct / total

    return epoch_loss, epoch_acc

In [ ]:
# ==========================================================
# Cell 40 : Validation
# ==========================================================

def validate(model, loader):

    model.eval()

    running_loss = 0

    correct = 0

    total = 0

    with torch.no_grad():

        for images, labels in tqdm(loader, leave=False):

            images = images.to(device)

            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            predictions = outputs.argmax(1)

            correct += (predictions == labels).sum().item()

            total += labels.size(0)

    epoch_loss = running_loss / len(loader)

    epoch_acc = 100 * correct / total

    return epoch_loss, epoch_acc

In [ ]:
# ==========================================================
# Cell 41 : Training Loop
# ==========================================================

for epoch in range(EPOCHS):

    print("\n" + "="*70)

    print(f"Epoch {epoch+1}/{EPOCHS}")

    print("="*70)

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader
    )

    val_loss, val_acc = validate(
        model,
        val_loader
    )

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print()

    print(f"Train Loss : {train_loss:.4f}")
    print(f"Train Acc  : {train_acc:.2f}%")

    print(f"Val Loss   : {val_loss:.4f}")
    print(f"Val Acc    : {val_acc:.2f}%")

    if val_acc > best_accuracy:

        best_accuracy = val_acc

        torch.save(
            model.state_dict(),
            "WeatherNet_Best.pth"
        )

        print()

        print("Best Model Saved")

print("\nTraining Finished.")

In [ ]:
# ==========================================================
# Cell 42 : Load Best Model
# ==========================================================

model.load_state_dict(
    torch.load(
        "WeatherNet_Best.pth",
        map_location=device
    )
)

model.eval()

print("Best Model Loaded")

In [ ]:
# ==========================================================
# Cell 43 : Training History
# ==========================================================

history = pd.DataFrame(history)

history.head()

In [ ]:
# ==========================================================
# Cell 44 : Loss Curve
# ==========================================================

plt.figure(figsize=(8,5))

plt.plot(
    history["train_loss"],
    label="Training"
)

plt.plot(
    history["val_loss"],
    label="Validation"
)

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.title("Training Loss")

plt.legend()

plt.grid()

plt.show()

In [ ]:
# ==========================================================
# Cell 45 : Accuracy Curve
# ==========================================================

plt.figure(figsize=(8,5))

plt.plot(
    history["train_acc"],
    label="Training"
)

plt.plot(
    history["val_acc"],
    label="Validation"
)

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.title("Accuracy")

plt.legend()

plt.grid()

plt.show()

In [ ]:
# ==========================================================
# Cell 46 : Final Results
# ==========================================================

print("="*60)

print(
    f"Best Validation Accuracy : {best_accuracy:.2f}%"
)

print("="*60)

In [ ]:
# ==========================================================
# Cell 47 : Evaluation Libraries
# ==========================================================

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [ ]:
# ==========================================================
# Cell 48 : Predict Validation Set
# ==========================================================

model.eval()

all_predictions = []

all_labels = []

with torch.no_grad():

    for images, labels in tqdm(val_loader):

        images = images.to(device)

        outputs = model(images)

        predictions = torch.argmax(outputs, dim=1)

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.numpy()
        )

print("Prediction Completed")

In [ ]:
# ==========================================================
# Cell 49 : Overall Metrics
# ==========================================================

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

precision = precision_score(
    all_labels,
    all_predictions,
    average="weighted"
)

recall = recall_score(
    all_labels,
    all_predictions,
    average="weighted"
)

f1 = f1_score(
    all_labels,
    all_predictions,
    average="weighted"
)

print("="*60)

print(f"Accuracy  : {accuracy:.4f}")

print(f"Precision : {precision:.4f}")

print(f"Recall    : {recall:.4f}")

print(f"F1 Score  : {f1:.4f}")

print("="*60)

In [ ]:
# ==========================================================
# Cell 50 : Classification Report
# ==========================================================

report = classification_report(

    all_labels,

    all_predictions,

    target_names=weather_classes,

    digits=4

)

print(report)

In [ ]:
# ==========================================================
# Cell 51 : Confusion Matrix
# ==========================================================

cm = confusion_matrix(

    all_labels,

    all_predictions

)

plt.figure(figsize=(8,8))

disp = ConfusionMatrixDisplay(

    confusion_matrix=cm,

    display_labels=weather_classes

)

disp.plot(
    cmap="Blues",
    xticks_rotation=45
)

plt.title("Confusion Matrix")

plt.show()

In [ ]:
# ==========================================================
# Cell 52 : Per-Class Accuracy
# ==========================================================

class_accuracy = cm.diagonal() / cm.sum(axis=1)

results = pd.DataFrame({

    "Weather": weather_classes,

    "Accuracy": class_accuracy

})

results

In [ ]:
# ==========================================================
# Cell 53 : Metrics Table
# ==========================================================

metrics = pd.DataFrame({

    "Metric":[

        "Accuracy",

        "Precision",

        "Recall",

        "F1 Score"

    ],

    "Value":[

        accuracy,

        precision,

        recall,

        f1

    ]

})

metrics

In [ ]:
# ==========================================================
# Cell 54 : Visualize Predictions
# ==========================================================

model.eval()

plt.figure(figsize=(15,15))

for i in range(9):

    idx = random.randint(
        0,
        len(val_dataset)-1
    )

    image, label = val_dataset[idx]

    input_image = image.unsqueeze(0).to(device)

    with torch.no_grad():

        output = model(input_image)

        prediction = torch.argmax(
            output,
            dim=1
        ).item()

    img = image.permute(1,2,0).numpy()

    img = img * np.array(
        [0.229,0.224,0.225]
    )

    img = img + np.array(
        [0.485,0.456,0.406]
    )

    img = np.clip(img,0,1)

    plt.subplot(3,3,i+1)

    plt.imshow(img)

    plt.title(

        f"GT : {weather_classes[label]}\n"

        f"Pred : {weather_classes[prediction]}"

    )

    plt.axis("off")

plt.tight_layout()

plt.show()

In [ ]:
# ==========================================================
# Cell 55 : Save Predictions
# ==========================================================

prediction_df = val_df.copy()

prediction_df["Prediction"] = [

    weather_classes[p]

    for p in all_predictions

]

prediction_df["GroundTruth"] = [

    weather_classes[l]

    for l in all_labels

]

prediction_df.to_csv(

    "WeatherNet_Predictions.csv",

    index=False

)

print("Predictions Saved")

In [ ]:
# ==========================================================
# Cell 56 : Save Evaluation Metrics
# ==========================================================

metrics.to_csv(

    "WeatherNet_Metrics.csv",

    index=False

)

results.to_csv(

    "PerClassAccuracy.csv",

    index=False

)

print("Evaluation Results Saved")

In [ ]:
# ==========================================================
# Cell 57 : Evaluation Summary
# ==========================================================

print("="*60)

print("WeatherNet Evaluation Completed")

print()

print(f"Validation Accuracy : {accuracy*100:.2f}%")

print(f"Weighted Precision  : {precision:.4f}")

print(f"Weighted Recall     : {recall:.4f}")

print(f"Weighted F1 Score   : {f1:.4f}")

print("="*60)

In [ ]:
# ==========================================================
# Cell 58 : Weather Knowledge Base
# ==========================================================

weather_rules = {

    "clear": {

        "risk": "Low",
        "recommended_speed": "80-100 km/h",
        "visibility": "Excellent",
        "decision": "Normal Driving"

    },

    "partly cloudy": {

        "risk": "Low",
        "recommended_speed": "70-90 km/h",
        "visibility": "Good",
        "decision": "Normal Driving"

    },

    "overcast": {

        "risk": "Low",
        "recommended_speed": "70-90 km/h",
        "visibility": "Moderate",
        "decision": "Normal Driving"

    },

    "foggy": {

        "risk": "High",
        "recommended_speed": "30-50 km/h",
        "visibility": "Poor",
        "decision": "Risk Driving"

    },

    "rainy": {

        "risk": "High",
        "recommended_speed": "40-60 km/h",
        "visibility": "Reduced",
        "decision": "Risk Driving"

    },

    "snowy": {

        "risk": "Very High",
        "recommended_speed": "20-40 km/h",
        "visibility": "Very Poor",
        "decision": "Risk Driving"

    }

}

print("Knowledge Base Loaded Successfully")

In [ ]:
# ==========================================================
# Cell 59 : Adaptive Decision Module
# ==========================================================

def adaptive_decision(weather, confidence):

    info = weather_rules[weather]

    return {

        "Predicted Weather": weather,

        "Confidence": f"{confidence:.2f}%",

        "Risk Level": info["risk"],

        "Visibility": info["visibility"],

        "Recommended Speed": info["recommended_speed"],

        "Decision": info["decision"]

    }

print("Adaptive Decision Module Ready")

In [ ]:
# ==========================================================
# Cell 60 : Single Image Prediction
# ==========================================================

model.eval()

image, label = val_dataset[25]

input_image = image.unsqueeze(0).to(device)

with torch.no_grad():

    output = model(input_image)

    probability = torch.softmax(output, dim=1)

    confidence, prediction = torch.max(probability, 1)

predicted_weather = weather_classes[prediction.item()]

confidence = confidence.item() * 100

print(predicted_weather)

print(confidence)

In [ ]:
# ==========================================================
# Cell 61 : Adaptive Decision
# ==========================================================

decision = adaptive_decision(

    predicted_weather,

    confidence

)

decision

In [ ]:
# ==========================================================
# Cell 62 : Visualization
# ==========================================================

display = image.permute(1,2,0).numpy()

display = display * np.array([0.229,0.224,0.225])

display = display + np.array([0.485,0.456,0.406])

display = np.clip(display,0,1)

plt.figure(figsize=(8,8))

plt.imshow(display)

plt.axis("off")

plt.title(

    f"Weather : {predicted_weather}\n"

    f"Confidence : {confidence:.2f}%\n"

    f"Decision : {decision['Decision']}"

)

plt.show()

In [ ]:
# ==========================================================
# Cell 63 : Batch Demonstration
# ==========================================================

results = []

model.eval()

for i in range(20):

    image, label = val_dataset[i]

    inp = image.unsqueeze(0).to(device)

    with torch.no_grad():

        out = model(inp)

        prob = torch.softmax(out, dim=1)

        conf, pred = torch.max(prob,1)

    weather = weather_classes[pred.item()]

    info = adaptive_decision(

        weather,

        conf.item()*100

    )

    results.append(info)

results = pd.DataFrame(results)

results

In [ ]:
# ==========================================================
# Cell 64 : Decision Distribution
# ==========================================================

plt.figure(figsize=(7,5))

results["Risk Level"].value_counts().plot(

    kind="bar",

    color=["green","orange","red","purple"]

)

plt.ylabel("Images")

plt.title("Adaptive Risk Levels")

plt.show()

In [ ]:
# ==========================================================
# Install Required Libraries
# ==========================================================

!pip install -q ultralytics
!pip install -q grad-cam
!pip install -q scikit-fuzzy

In [ ]:
# ==========================================================
# Cell 65 : Load YOLO11
# ==========================================================

from ultralytics import YOLO

yolo = YOLO("yolo11n.pt")

print("YOLO11 Loaded Successfully")

In [ ]:
# ==========================================================
# Cell 66 : Feature Extraction Module
# ==========================================================

feature_extractor = yolo.model.model[:-1]

print(feature_extractor)

In [ ]:
# ==========================================================
# Cell 67 : Select Random Image
# ==========================================================

import os
import random

IMAGE_DIR = "/kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k/bdd100k/images/100k/val"

image_list = [
    img for img in os.listdir(IMAGE_DIR)
    if img.endswith((".jpg", ".jpeg", ".png"))
]

IMAGE_NAME = random.choice(image_list)

IMAGE_PATH = os.path.join(IMAGE_DIR, IMAGE_NAME)

print("Selected Image:")
print(IMAGE_NAME)
print(IMAGE_PATH)

In [ ]:
# ==========================================================
# Cell 68 : Display Random Image
# ==========================================================

import cv2
import matplotlib.pyplot as plt

img = cv2.imread(IMAGE_PATH)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(8,8))
plt.imshow(img)
plt.title(IMAGE_NAME)
plt.axis("off")
plt.show()

In [ ]:
# ==========================================================
# Cell 69 : YOLO11 Detection
# ==========================================================

results = yolo.predict(
    source=IMAGE_PATH,
    conf=0.25,
    verbose=False
)

results[0].show()

In [ ]:
# ==========================================================
# Cell 70 : Detection Results
# ==========================================================

boxes = results[0].boxes

print(f"Objects detected in {IMAGE_NAME}\n")

for box in boxes:

    cls = int(box.cls.item())
    conf = float(box.conf.item())

    print(f"{yolo.names[cls]:20}  Confidence: {conf:.3f}")

In [ ]:
# ==========================================================
# Cell 71 : Adaptive Weather Decision Module
# ==========================================================

def adaptive_weather_controller(weather, confidence):

    confidence = confidence / 100.0

    if weather == "clear":
        return {
            "attention":0.30,
            "heatmap":0.20,
            "fusion":0.30,
            "gain":1.00
        }

    elif weather == "cloudy":
        return {
            "attention":0.45,
            "heatmap":0.35,
            "fusion":0.45,
            "gain":1.10
        }

    elif weather == "partly cloudy":
        return {
            "attention":0.50,
            "heatmap":0.40,
            "fusion":0.50,
            "gain":1.15
        }

    elif weather == "rainy":
        return {
            "attention":0.80,
            "heatmap":0.90,
            "fusion":0.85,
            "gain":1.40
        }

    elif weather == "foggy":
        return {
            "attention":0.95,
            "heatmap":1.00,
            "fusion":0.95,
            "gain":1.60
        }

    elif weather == "snowy":
        return {
            "attention":0.90,
            "heatmap":0.95,
            "fusion":0.90,
            "gain":1.50
        }

    else:
        return {
            "attention":0.50,
            "heatmap":0.50,
            "fusion":0.50,
            "gain":1.00
        }

In [ ]:
# ==========================================================
# Cell 72 : Test Adaptive Controller
# ==========================================================

weather = "rainy"
confidence = 92

params = adaptive_weather_controller(
    weather,
    confidence
)

params

In [ ]:
# ==========================================================
# Cell 73 : WeatherNet → Controller
# ==========================================================

image, label = val_dataset[0]

image = image.unsqueeze(0).to(device)

model.eval()

with torch.no_grad():

    output = model(image)

    probability = torch.softmax(output,1)

    confidence, prediction = torch.max(probability,1)

predicted_weather = weather_classes[prediction.item()]

confidence = confidence.item()*100

print("Weather :", predicted_weather)
print("Confidence :", round(confidence,2))

In [ ]:
# ==========================================================
# Cell 74 : Adaptive Parameters
# ==========================================================

adaptive_parameters = adaptive_weather_controller(
    predicted_weather,
    confidence
)

adaptive_parameters

In [ ]:
import ultralytics

print("Ultralytics Version:", ultralytics.__version__)

print("\nYOLO Model Structure:\n")

for i, layer in enumerate(yolo.model.model):
    print(i, layer.__class__.__name__)

In [179]:
import os

print(os.path.exists("/kaggle/working/WeatherNet_Best.pth"))

if os.path.exists("/kaggle/working/WeatherNet_Best.pth"):
    print("File Size:", os.path.getsize("/kaggle/working/WeatherNet_Best.pth")/1024/1024, "MB")

True
File Size: 10.68595027923584 MB
